In [1]:
import numpy as np
import scipy as sp
import regex as re
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
print(jax.devices())
import jaxlib
import jax.numpy as jnp
import flax
import flax.linen as nn
import optax
from typing import Tuple, Callable, Any, Dict, Optional
import numpy.typing as npt
import copy
import pathlib
import matplotlib.pyplot as plt
import time
import json
import ast
import netket as nk
import os
import glob
import sys

sys.path.append("/home/ihuarte/Escritorio/Ivan/NNs")

# os.chdir("/home/ihuarte/Escritorio/Ivan/NN/")

from VA_project.model.model import J1J2Square, Oxalate
from VA_project.engine.runners import Runner

# from NN_utils import load_vstate
# from correlations import correlations_vstate

# from NNs.NN_module.ST_utils import compare_params, masked_optimizer
from frozendict import deepfreeze
from NN_module.ST_utils import print_tree

[CpuDevice(id=0)]


∣NK⟩ Tip: Debug multi-node HPC? `djaxrun -np 2 python Examples/Sharding/multi_process.py`

Call to org.freedesktop.portal.Settings.ReadAll failed QDBusError("org.freedesktop.DBus.Error.NoReply", "Did not receive a reply. Possible causes include: the remote application did not send a reply, the message bus security policy blocked the reply, the reply timeout expired, or the network connection was broken.")
/home/ihuarte/miniconda3/envs/conda_env/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/jax_array_handlers.py:711: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(
qt.svg: Cannot open file '/home/ihuarte/miniconda3/envs/conda_env/lib/python3.12/site-packages/matplotlib/mpl-data/images/matplotlib.svg', because: No such file or directory


qt.qpa.theme.gnome: dbus reply error: [ "org.freedesktop.DBus.Error.NoReply" ] "Did not receive a reply. Possible causes include: the remote application did not send a reply, the message bus security policy blocked the reply, the reply timeout expired, or the network connection was broken."
qt.qpa.theme.gnome: dbus reply error: [ "org.freedesktop.DBus.Error.NoReply" ] "Did not receive a reply. Possible causes include: the remote application did not send a reply, the message bus security policy blocked the reply, the reply timeout expired, or the network connection was broken."

** (python:668721): WARNING **: 12:00:22.355: atk-bridge: get_device_events_reply: unknown signature


In [ ]:
def c2_operation(lattice_size):
    def _operation(configuration):
        configuration = configuration.reshape(lattice_size)
        configuration = configuration.T[::-1].T[::-1]
        return configuration.reshape(-1)

    return _operation

c4_operation(size)(x).reshape(size)

In [ ]:
size = (5,5)
N = size[0] * size[1]
x = jnp.arange(N)[None, :].reshape(size)

def rotate60_safe(lattice):
    Lx, Ly = lattice

    def nruter(x_in):
        rotated = jnp.zeros(lattice, dtype=jnp.int64)
        for i in range(Lx):
            for j in range(Ly):

                # coords reales
                x = i + 0.5 * j 
                y = (jnp.sqrt(3)/2) * j 

                # rotación 60°
                x_r = 0.5 * x - (jnp.sqrt(3)/2) * y 
                y_r = (np.sqrt(3)/2) * x + 0.5 * y 

                # volver a coords discretas
                j_new = int(round(y_r / (jnp.sqrt(3)/2)))
                i_new = int(round(x_r - 0.5 * j_new))

                i_new %= Lx
                j_new %= Ly
                

                rotated = rotated.at[i_new, j_new].set(x_in[i, j])

        return rotated
    return nruter

perm = rotate60_safe(size)
print(jnp.roll(x, shift=(2,2), axis=(0,1)))
y = perm(x)
jnp.roll(y, shift=(2,2), axis=(0,1))

In [ ]:
y

In [ ]:
jnp.roll(y, shift=(2,2), axis=(0,1))


In [ ]:
%run ./VMC_simulation.py

In [ ]:
B = 1
size=(6,6)
N = size[0]*size[1]

x = jnp.arange(N)[None, :]
x = jnp.ones(N)[None, :]
x = jnp.broadcast_to(x ,(B, N))

x = x.at[:,0].set(-1.0)

In [ ]:
vstate.model.apply(vstate.variables, x)

In [ ]:
a = jnp.ones((2,36))
b = jnp.arange(36)


In [ ]:
c = a*b
c, c.shape

In [ ]:
raw = jnp.array(
[[[10.61138765+22.84845978j, 11.61624836+23.58600621j,
   10.61138765+22.84845978j, 11.61624836+23.58600621j,
   10.61138765+22.84845978j, 11.61624836+23.58600621j],
  [11.37925431+23.38226749j, 11.06942364+23.10205822j,
   11.37925431+23.38226749j, 11.06942364+23.10205822j,
   11.37925431+23.38226749j, 11.06942364+23.10205822j],
  [10.61138765+22.84845978j, 11.61624836+23.58600621j,
   10.61138765+22.84845978j, 11.61624836+23.58600621j,
   10.61138765+22.84845978j, 11.61624836+23.58600621j],
  [11.37925431+23.38226749j, 11.06942364+23.10205822j,
   11.37925431+23.38226749j, 11.06942364+23.10205822j,
   11.37925431+23.38226749j, 11.06942364+23.10205822j],
  [10.61138765+22.84845978j, 11.61624836+23.58600621j,
   10.61138765+22.84845978j, 11.61624836+23.58600621j,
   10.61138765+22.84845978j, 11.61624836+23.58600621j],
  [11.37925431+23.38226749j, 11.06942364+23.10205822j,
   11.37925431+23.38226749j, 11.06942364+23.10205822j,
   11.37925431+23.38226749j, 11.06942364+23.10205822j]]], dtype=jnp.complex128
)

# real = jnp.where(raw.real > 1e-5, raw.real, 0.0)
# imag = jnp.where(raw.imag > 1e-7, raw.imag, 0.0)
# b = real + 1.0j * imag
# mod = jnp.abs(b)
# phase = jnp.angle(b)
# mod,phase/jnp.pi*180
# raw = raw - raw[0,0,0]
real = raw.real
imag = raw.imag

In [ ]:
raw

In [ ]:
%matplotlib inline
_, ax = plt.subplots(1, figsize=[10, 6])
ax.scatter(real.flatten(), imag.flatten(),)
ax.plot()
ax.set_xlabel("Real(z)")
ax.set_ylabel("Imag(z)")
ax.grid()
plt.show()

In [ ]:
mod = jnp.abs(raw)
phase = jnp.angle(raw)
i, j = jnp.unravel_index(jnp.arange(36), (6,6))
indices = jnp.array([i,j]).T.tolist()
indices = [str(ind) for ind in indices]
indices

In [ ]:



_, (ax1, ax2) = plt.subplots(2,1, figsize=[16, 16])
ax1.plot(range(36), mod.flatten(), ls='', marker='o', ms=3)
ax1.set_xticks(range(36))
ax1.set_xticklabels(indices, rotation=45, ha="right")
ax1.set_xlabel("Traslacion(z)")
ax1.set_ylabel("Modulo(z)")
ax1.grid()

ax2.plot(range(36), phase.flatten(), ls='', marker='o', ms=3)
ax2.set_xticks(range(36))
ax2.set_xticklabels(indices, rotation=45, ha="right")
ax2.set_xlabel("Traslacion(z)")
ax2.set_ylabel("Fase(z)")
ax2.set_ylim(-jnp.pi, jnp.pi)
ax2.grid()

axin = ax2.inset_axes(bounds=(0.25, 0.1, 0.4, 0.3) )
axin.plot(range(36), phase.flatten(), ls='', marker='o', ms=3)
axin.grid()

plt.show()

In [ ]:
from NN_module.NN.utils import setup_from_template
from NN_module.ST_utils import print_tree

template = config_nn[config_nn["selection"]]["stage0"]["template"]
storage = config_nn["storage"]
symm_wrapper = config_nn["symm_wrapper"]
print_tree(setup_from_template(template, storage, symm_wrapper), values=True)

hydra = Hydra(config_nn, **{"lattice_size": size})
model = hydra.model
params = model.init(jax.random.PRNGKey(0), jnp.ones((1, 36)))

code2path = hydra.get_code2path(params["params"])
code2path



In [ ]:
from NN_module.schedule.schedule import Schedule
#### INITIALIZE SCHEDULE ####
schedule_setup = config["schedule"]
schedule = Schedule(schedule_setup, code2path)
total_periods = schedule.total_periods
total_epochs = schedule.total_epochs
schedule_setup["total_epochs"] = total_epochs

In [ ]:

for i, (lr_period, info, change) in enumerate(schedule.schedule()):
    
    #### PRINT PERIOD INFO ####
    epochs = info[0][0]
    mode = info[0][1]
    lr_string = ""
    for inf in info:
        lr_string += f"{inf[2]}  "
    rescaled = "" if len(info) < 4 else info[3]
    print(f"\nPeriod {i + 1} / {total_periods}:")
    print(f"Training {mode} for {epochs} epochs")
    print(f"LR: {lr_string}  ({rescaled})")
    ###########################

    optimizer = schedule.transform_optimizer(
        params['params'], optax.sgd, info, lr_period
    )
    print(optimizer)


In [ ]:
schedule.eon_code2path

In [ ]:
from NN_module.NN import factory_submodule_tags, __all_single__
def get_code2path_tree(params, wraps=[], prefix=""):
    """
    Recursively builds a tree from a Flax/JAX pytree of parameters,
    assigning a unique positional index to each node.

    Each node is stored as:
        tree[name] = {
            "_idx": <hierarchical code>,
            "_children": { ... }
        }
    Indices are generated by concatenating child positions at each level:
        "0", "01", "010", etc.
    """
    if not isinstance(params, dict):
        # Hoja, retornamos vacío
        return {}
    
    

    tree = {}
    for i, (name, subtree) in enumerate(params.items()):
        if name in symm_submodule_dict.values():
            idx = f"{prefix}"
        else:
            idx = f"{prefix}{i}"
        tree[name] = {"_idx": idx, "_children": get_code2path_tree(subtree, prefix=idx)}
    return tree

def is_main_module(name):
    if "_" in name:
        name = name.split("_")[0]
    return name in factory_submodule_tags


def is_simple_module(raw_name):
    raw_name = raw_name[0]
    if "_" in raw_name:
        raw_name = raw_name.split("_")[0]
    if "Worker" in raw_name:
        raw_name = raw_name.split("Worker")[0]
    if "Block" in raw_name:
        raw_name = raw_name.split("Block")[0]

    if raw_name in __all_single__:
        return f"({raw_name})"
    else:
        return ""
def is_wrap(name):
    return name in symm_submodule_dict.values()

def print_architecture(tree, print_all=True, prefix="", is_last=True, wraps=[]):

    keys = list(tree.keys())

    for i, name in enumerate(keys):
        data = tree[name]
        last = i == len(keys) - 1
       
        iswrap = is_wrap(name)
        
        if iswrap:
             # Is Symm_Wrap
             
            children = list(data["_children"].keys())
            if len(children) == 1:
                child_name = children[0]
                if is_wrap(child_name):
                    print_architecture(data["_children"], print_all, prefix, is_last, wraps+[name])
                else:
                    wraps.append(name)
                    wrap_label = " ".join(wraps[1:]) if len(wraps) > 1 else print("No symmetrization")
                    print(f"SymmModel: ({wrap_label})")  if len(wraps) > 1 else print("SymmModel:")
                    print_architecture(data["_children"], print_all, prefix, is_last)
            else:
                wraps.append(name)
                wrap_label = " ".join(wraps[1:]) if len(wraps) > 1 else print("No symmetrization")
                print(f"SymmModel: ({wrap_label})")  if len(wraps) > 1 else print("SymmModel:")
                print_architecture(data["_children"], print_all, prefix, is_last)

        else:
                
            # Is factory
            is_main = is_main_module(name)
            children = list(data["_children"].keys())
            simple_module = ""
            if is_main:
                if len(children) == 1:
                    simple_module = is_simple_module(children)
                else:
                    candidates = [is_simple_module([child]) for child in children]
                    simple_module = next((c for c in candidates if c != ""), "")
            
            if not print_all and not is_main:
                continue

            annotation = ""
            if is_main and simple_module:
                annotation += f"{simple_module}"

            connector = "└── " if last else "├── "
            print(prefix + connector + f"{name} {annotation}  --->  ({data['_idx']})")

            if data["_children"] and not iswrap:
                new_prefix = prefix + ("    " if last else "│   ")
                print_architecture(
                    data["_children"], print_all, prefix=new_prefix, is_last=last
                )


In [ ]:
tree = get_code2path_tree(params["params"])
print_architecture(tree, print_all=False)

In [ ]:
print_tree(params)

In [ ]:

def c4_operation(lattice_size):
    def _operation(configuration):
        configuration = configuration.reshape(lattice_size).T[::-1]
        return configuration.reshape(-1)
    return _operation

class Cn():

    def __init__(self, lattice_size, n):
        """
        C_n rotation symmetry
        """

        self.lattice_size = lattice_size

        self.dim = 1
        self.N = 4
        self.character = lambda q, m: jnp.exp(2 * jnp.pi * q * m / n)

        assert self.N != 1, f"Group size too low. N = 1"
        assert all(L != 1 for L in self.lattice_size)

        if n == 4:
            self.operation = c4_operation(lattice_size)
        

    
class Refl_H():
    
    def __init__(self, lattice_size):
        """
        Reflection symmetry respect to the horizontal axis
        """

        self.lattice_size = lattice_size

        self.dim = 1
        self.N = 2
        self.character = lambda k, m: 1 if k==0 else (-1)**m

        assert self.N != 1, f"Group size too low. N = 1"
        assert all(L != 1 for L in self.lattice_size)

    def operation(self, configuration):
        return configuration.reshape(self.lattice_size)[::-1]
    
class Refl_V():
    
    def __init__(self, lattice_size):
        """
        Reflection symmetry respect to the horizontal axis
        """

        self.lattice_size = lattice_size

        self.dim = 1
        self.N = 2
        self.character = lambda k, m: 1 if k==0 else (-1)**m

        assert self.N != 1, f"Group size too low. N = 1"
        assert all(L != 1 for L in self.lattice_size)

    def operation(self, configuration):
        return configuration.reshape(self.lattice_size)[:, ::-1]

class Refl_D():
    
    def __init__(self, lattice_size):
        """
        Reflection symmetry respect to the horizontal axis
        """

        self.lattice_size = lattice_size

        self.dim = 1
        self.N = 2
        self.character = lambda k, m: 1 if k==0 else (-1)**m


        assert self.N != 1, f"Group size too low. N = 1"
        assert all(L != 1 for L in self.lattice_size)

    def operation(self, configuration):
        return configuration.reshape(self.lattice_size).T
    
class Refl_AD():
    
    def __init__(self, lattice_size):
        """
        Reflection symmetry respect to the horizontal axis
        """

        self.lattice_size = lattice_size

        self.dim = 1
        self.N = 2
        self.character = lambda k, m: 1 if k==0 else (-1)**m

        assert self.N != 1, f"Group size too low. N = 1"
        assert all(L != 1 for L in self.lattice_size)

    def operation(self, configuration):
        return configuration.reshape(self.lattice_size)[:, ::-1].T[:, ::-1]
    

In [ ]:
size = (3,3)
N = jnp.prod(jnp.array(size))
x = jnp.arange(N)
sym = Refl_AD(size)
x.reshape(size)

In [ ]:
x_1 = sym.operation(x)
x_1

In [ ]:
x_2 = sym.operation(x_1)
x_2

In [ ]:
x_3 = sym.operation(x_2)
x_3

In [ ]:
x_3 = sym.operation(x_3)
x_3

In [ ]:
        print(f"Subtype: {jnp.issubdtype(jnp.complex128, jnp.complexfloating)}")


In [ ]:
import numpy as np
#oxalate
x_ED = np.loadtxt("/home/ihuarte/Escritorio/Ivan/NNs/new_pruebas/Oxalate/Factorized_CvT_Szabo_Transversal_Factorized/Size_4x4/ST_1000-A/UUID_4e71c27c/Oxalate_Factorized_CvT_Szabo_xED_4x4_strength_0.2_theta_9.0_phi_72.0.txt", dtype=complex)

# j1j2
# x_ED = np.loadtxt("/home/ihuarte/Escritorio/Ivan/NNs/new_pruebas/J1J2Square/Jastrow_Transversal_Jastrow_wrap/Size_4x4/ST_1500A/UUID_1b2e9d23/J1J2Square_Jastrow_xED_4x4_J1J2_1.0_0.5_XYZ_0.0_0.0_0.0.txt", dtype=complex)


In [ ]:
size = (4,3)
a =0.2
theta = 9.0
phi = 72.0
cm_model = Oxalate(size, [a, theta, phi], order="default_2", bc='periodic')

eng = Runner(cm_model.cm, S_operators=True)
E_ED, x_ED = eng.exact_energy_lanczos(k=1, eigenstates=True)
x_ED=x_ED.flatten()

In [ ]:
E_ED

In [ ]:
mod = jnp.abs(x_ED)
phase = jnp.angle(x_ED)

no_null_mod_mask = mod > 1e-12
phase_mask = phase[no_null_mod_mask]

counts, values = jnp.histogram(phase, bins=1000000)
counts_mask, values_mask = jnp.histogram(phase_mask, bins=1000000)

weights = mod
counts_weighted, values_weighted = jnp.histogram(phase_mask, bins=1000000, weights=weights)


In [ ]:
kde = sp.stats.gaussian_kde(phase, bw_method=0.01, weights=weights/jnp.sum(weights))

In [ ]:
_, ax = plt.subplots(figsize=(15,6))
ax.set_title("Phase distribution KDE size " + f"{size[0]}x{size[1]}", fontsize=20)
ax.plot(jnp.linspace(-jnp.pi, jnp.pi, 1000), kde.evaluate(jnp.linspace(-jnp.pi, jnp.pi, 1000))[::-1], label="KDE with weights", color="blue")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 1,  figsize=[20, 10])
fig.suptitle("J1-J2 Model  (J2/J1 = 0.5)", fontsize=30)
fig.suptitle("Oxalate  (QSL)", fontsize=30)
ax[0].set_title("Phase histogram")
ax[1].set_title("Phase histogram with no null modulus")
ax[2].set_title("Phase histogram weighted by modulus squared")
ax[0].hist(
    phase,
    bins=1000,
    range=(-np.pi, np.pi),
    density=True,
    alpha=0.7,
    label=f"ED",
)
ax[1].hist(
    phase_mask,
    bins=1000,
    range=(-np.pi, np.pi),
    density=True,
    alpha=0.7,
    label=f"ED",
)
ax[2].hist(
    phase,
    bins=1000,
    range=(-np.pi, np.pi),
    weights=weights,
    density=True,
    alpha=0.7,
    label=f"ED",
)

ax[0].set_yscale('log')
ax[1].set_yscale('log')
ax[2].set_yscale('log')
ax[0].set_ylim(1e-2, 1e2)
ax[1].set_ylim(1e-2, 1e2)
ax[2].set_ylim(1e-2, 1e2)

In [ ]:
from NN_module.observables import all_spin_configurations

In [ ]:
def print_max_contributors(x, size, N_max=10, return_states=False):
    (mod, ph), _ = modphase(x)
    configs = all_spin_configurations(size[0] * size[1])

    idx = jnp.argsort(mod)[::-1][:N_max]
    max_configs = configs[idx, :]
    max_mods = mod[idx]
    max_phs = ph[idx]

    for config, mod, phs in zip(max_configs, max_mods, max_phs):
        tmagn = jnp.sum(config.flatten())
        print(f"Config: \n{config.reshape(size)}")
        print(f"M = {tmagn}")
        print(f"\nModulus: {mod}")
        print(f"Phase: {phs}\n")

    if return_states:
        return max_configs

In [ ]:
size = (4,4)
mod = jnp.abs(x_ED)
phase = jnp.angle(x_ED)
configs = all_spin_configurations(size[0] * size[1])

max_contr_idx = jnp.argsort(mod)[::-1]
sorted_modulus = mod[max_contr_idx]
sorted_configs = configs[max_contr_idx, :]

config_magnetization = jnp.sum(sorted_configs, axis=-1)/2



In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1,  figsize=[20, 10])

fig.suptitle("J1-J2 Model  (J2/J1 = 0.5)", fontsize=30)
fig.suptitle("Oxalate  (QSL)", fontsize=30)

ax[0].set_title("Magnetization histogram (raw histogram)")
ax[1].set_title("Magnetization histogram (weighted by modulus)")
ax[0].hist(
    config_magnetization,
    bins=33,
    range=(-16, 16),
    density=True,
    alpha=0.7,
    label=f"ED",
    edgecolor='black',

)
ax[1].hist(
    config_magnetization,
    bins=33,
    range=(-16, 16),
    weights=sorted_modulus,
    density=True,
    alpha=0.7,
    label=f"ED",
    edgecolor='black',
    
)

ax[0].set_xticks(jnp.arange(-16,18,2))
ax[1].set_xticks(jnp.arange(-16,18,2))
# ax[0].set_yscale('log')
# ax[1].set_yscale('log')
# ax[0].set_ylim(1e-2, 1e2)
# ax[1].set_ylim(1e-2, 1e2)

In [ ]:
max(mod), min(mod)

In [ ]:
tramos = jnp.logspace(-6, 0, 100)
perc_under_x = []

for x in tramos:
    perc_under_x.append((sorted_modulus < x).sum()/sorted_modulus.shape[0] * 100)

    
_, ax = plt.subplots(figsize=(10,6))
ax.plot(tramos, perc_under_x)
ax.set_title("Percentage of total components under modulus ", fontsize=20)
ax.set_xlabel("Modulus", fontsize=15)
ax.set_ylabel("Percentage", fontsize=15)
ax.set_xscale('log')

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy as sp
from VA_project.model.model import J1J2Square, Oxalate
from VA_project.engine.runners import Runner

size = (4,3)
a =0.2
theta = 9.0
phi = 72.0

for size in [(2,2), (2,3), (3,3), (4,3), (4,4), (4,5)]:
    cm_model = Oxalate(size, [a, theta, phi], order="default_2", bc='periodic')

    eng = Runner(cm_model.cm, S_operators=True)
    E_ED, x_ED = eng.exact_energy_lanczos(k=1, eigenstates=True)
    x_ED=x_ED.flatten()
    mod = jnp.abs(x_ED)
    phase = jnp.angle(x_ED)
    weights = mod
    kde = sp.stats.gaussian_kde(phase, bw_method=0.01, weights=weights/jnp.sum(weights))

    _, ax = plt.subplots(figsize=(15,6))
    ax.set_title("Phase distribution KDE size " + f"{size[0]}x{size[1]}", fontsize=20)
    ax.plot(jnp.linspace(-jnp.pi, jnp.pi, 1000), kde.evaluate(jnp.linspace(-jnp.pi, jnp.pi, 1000))[::-1], label="KDE with weights", color="blue")
    plt.show()
    plt.close()

In [ ]:
import glob
from NN_module.saveNload import load_pytree
from NN_module.observables import distance_with_neel, distance_with_stripped
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import jax.numpy as jnp
size = (10,10)
neel = distance_with_neel(size)
stripped = distance_with_stripped(size)

parent = "/home/ihuarte/Escritorio/Ivan/NNs/pruebas/Neel/"
paths_list = glob.glob(parent + "**/*_results_*.json", recursive=True)

for path in paths_list:

    with open(path, 'r') as f:
        artifact = json.load(f)

    param_path =artifact["_artifacts"]["parameters"]
    callback_path = artifact["_artifacts"]["callback"]["plot"]

    img = mpimg.imread(callback_path)
    plt.figure(figsize=(15,15))
    plt.imshow(img)
    plt.axis('off')  
    plt.show()

    parameters = load_pytree(param_path)
    print("\n")
    print(f"File: {path}")
    print(parameters)
    parameters = parameters['Trans_0']['phi'].reshape(size)
    filtered_params = jnp.where(parameters < 0, -1, 1)
    print(filtered_params) 

    dist_neel, neel_label = neel(filtered_params)
    dist_strip, strip_label = stripped(filtered_params)

    print("\n")
    print(f"Distance with Neel:     {dist_neel} ({neel_label})")
    print(f"Distance with Stripped: {dist_strip} ({strip_label})")

    print("\n")